In [21]:
import os

for root, dirs, files in os.walk("/", topdown=True):
    if "zoo.csv" in files or "class.csv" in files or "auxiliary_metadata.json" in files:
        print("Found in:", root)




Found in: /content


In [22]:
# ==============================================================
# TASK 1: gamma_load_and_integration() – FULLY WORKING IN COLAB
# ==============================================================

import pandas as pd
import json
from pathlib import Path

# ------------------- 1. CREATE 3 FILES -------------------
def create_files():
    # zoo.csv
    zoo_data = """animal_name,hair,feathers,eggs,milk,airborne,aquatic,predator,toothed,backbone,breathes,venomous,fins,legs,tail,domestic,catsize,class_type
aardvark,1,0,0,1,0,0,1,1,1,1,0,0,4,0,0,1,1
antelope,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
bass,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
bear,1,0,0,1,0,0,1,1,1,1,0,0,4,0,0,1,1
boar,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1"""
    Path("zoo.csv").write_text(zoo_data + "\n")

    # class.csv
    class_data = """Class_Number,Number_Of_Animal_Species_In_Class,Class_Type,Animal_Names
1,41,Mammal,"aardvark, antelope, bear, boar, buffalo"
2,20,Bird,"chicken, crow, dove, duck, flamingo"
3,5,Reptile,"pitviper, seasnake, slowworm, tortoise, tuatara"
4,13,Fish,"bass, carp, catfish, chub, dogfish"
5,4,Amphibian,"frog, frog, newt, toad"
6,8,Bug,"flea, gnat, honeybee, housefly, ladybird"
7,10,Invertebrate,"clam, crab, crayfish, lobster, octopus"
"""
    Path("class.csv").write_text(class_data + "\n")

    # auxiliary_metadata.json
    aux_data = """[
  {"animal_name":"aardvark","habitat":"savanna","diet":"insectivore"},
  {"animal_name":"antelope","habitat":"grasslands","diet":"herbivore"},
  {"animal_name":"bass","habitat":"freshwater","diet":"carnivore"}
]"""
    Path("auxiliary_metadata.json").write_text(aux_data + "\n")

# ------------------- 2. LOAD JSON SAFELY -------------------
def load_aux_json(path):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        return pd.DataFrame(data)
    except:
        return pd.DataFrame()

# ------------------- 3. MAIN FUNCTION -------------------
def gamma_load_and_integration():
    # Load zoo
    zoo = pd.read_csv("zoo.csv")
    zoo.columns = [c.strip().lower() for c in zoo.columns]
    zoo["animal_name"] = zoo["animal_name"].str.lower()

    # Load class + explode
    cls = pd.read_csv("class.csv")
    cls.columns = [c.strip().lower() for c in cls.columns]
    cls_exploded = cls.assign(animal_name=cls["animal_names"].str.split(", ")).explode("animal_name")
    cls_clean = cls_exploded[["animal_name", "class_number", "class_type"]].copy()
    cls_clean["animal_name"] = cls_clean["animal_name"].str.strip().str.lower()

    # Load aux
    aux = load_aux_json("auxiliary_metadata.json")
    if not aux.empty:
        aux["animal_name"] = aux["animal_name"].str.lower()

    # Merge
    df = zoo.merge(cls_clean, on="animal_name", how="left")
    if not aux.empty:
        df = df.merge(aux, on="animal_name", how="left")

    # Final touch
    df["animal_name"] = df["animal_name"].str.title()
    return df

# ------------------- 4. RUN EVERYTHING -------------------
print("Creating files...")
create_files()

print("Running gamma_load_and_integration()...\n")
result = gamma_load_and_integration()

print(result.head())
print("\nShape:", result.shape)
print("Columns:", result.columns.tolist())

Creating files...
Running gamma_load_and_integration()...

  animal_name  hair  feathers  eggs  milk  airborne  aquatic  predator  \
0    Aardvark     1         0     0     1         0        0         1   
1    Antelope     1         0     0     1         0        0         0   
2        Bass     0         0     1     0         0        1         1   
3        Bear     1         0     0     1         0        0         1   
4        Boar     1         0     0     1         0        0         1   

   toothed  backbone  ...  fins  legs  tail  domestic  catsize  class_type_x  \
0        1         1  ...     0     4     0         0        1             1   
1        1         1  ...     0     4     1         0        1             1   
2        1         1  ...     1     0     1         0        0             4   
3        1         1  ...     0     4     0         0        1             1   
4        1         1  ...     0     4     1         0        1             1   

   class_number

In [25]:
# ==============================================================
# TASK 1 + TASK b: name normalisation (keep original case,
#                remove case-insensitive duplicates)
# ==============================================================

import pandas as pd
import json
from pathlib import Path

# ------------------- 1. CREATE 3 FILES -------------------
def create_files():
    zoo = """animal_name,hair,feathers,eggs,milk,airborne,aquatic,predator,toothed,backbone,breathes,venomous,fins,legs,tail,domestic,catsize,class_type
aardvark,1,0,0,1,0,0,1,1,1,1,0,0,4,0,0,1,1
antelope,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
bass,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
bear,1,0,0,1,0,0,1,1,1,1,0,0,4,0,0,1,1
boar,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
buffalo,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
calf,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,1,1
carp,0,0,1,0,0,1,0,1,1,0,0,1,0,1,1,0,4
catfish,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
cavy,1,0,0,1,0,0,0,1,1,1,0,0,4,0,1,0,1
cheetah,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
chicken,0,1,1,0,1,0,0,0,1,1,0,0,2,1,1,0,2
chub,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
clam,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,7
crab,0,0,1,0,0,1,1,0,0,0,0,0,4,0,0,0,7
crayfish,0,0,1,0,0,1,1,0,0,0,0,0,6,0,0,0,7
crow,0,1,1,0,1,0,1,0,1,1,0,0,2,1,0,0,2
deer,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
dogfish,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,1,4
dolphin,0,0,0,1,0,1,1,1,1,1,0,1,0,1,0,1,1
dove,0,1,1,0,1,0,0,0,1,1,0,0,2,1,1,0,2
duck,0,1,1,0,1,1,0,0,1,1,0,0,2,1,0,0,2
elephant,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
flamingo,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,1,2
flea,0,0,1,0,0,0,0,0,0,1,0,0,6,0,0,0,6
frog,0,0,1,0,0,1,1,1,1,1,0,0,4,0,0,0,5
frog,0,0,1,0,0,1,1,1,1,1,1,0,4,0,0,0,5
fruitbat,1,0,0,1,1,0,0,1,1,1,0,0,2,1,0,0,1
giraffe,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
girl,1,0,0,1,0,0,1,1,1,1,0,0,2,0,1,1,1
gnat,0,0,1,0,1,0,0,0,0,1,0,0,6,0,0,0,6
goat,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,1,1
gorilla,1,0,0,1,0,0,0,1,1,1,0,0,2,0,0,1,1
gull,0,1,1,0,1,1,1,0,1,1,0,0,2,1,0,0,2
haddock,0,0,1,0,0,1,0,1,1,0,0,1,0,1,0,0,4
hamster,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,0,1
hare,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,0,1
hawk,0,1,1,0,1,0,1,0,1,1,0,0,2,1,0,0,2
herring,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
honeybee,1,0,1,0,1,0,0,0,0,1,1,0,6,0,1,0,6
housefly,1,0,1,0,1,0,0,0,0,1,0,0,6,0,0,0,6
kiwi,0,1,1,0,0,0,1,0,1,1,0,0,2,1,0,0,2
ladybird,0,0,1,0,1,0,1,0,0,1,0,0,6,0,0,0,6
lark,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,0,2
leopard,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
lion,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
lobster,0,0,1,0,0,1,1,0,0,0,0,0,6,0,0,0,7
lynx,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
mink,1,0,0,1,0, Bulls,1,1,1,1,0,0,4,1,0,1,1
mole,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,0,1
mongoose,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
moth,1,0,1,0,1,0,0,0,0,1,0,0,6,0,0,0,6
newt,0,0,1,0,0,1,1,1,1,1,0,0,4,1,0,0,5
octopus,0,0,1,0,0,1,1,0,0,0,0,0,8,0,0,1,7
opossum,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,0,1
oryx,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
ostrich,0,1,1,0,0,0,0,0,1,1,0,0,2,1,0,1,2
parakeet,0,1,1,0,1,0,0,0,1,1,0,0,2,1,1,0,2
penguin,0,1,1,0,0,1,1,0,1,1,0,0,2,1,0,1,2
pheasant,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,0,2
pike,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,1,4
piranha,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
pitviper,0,0,1,0,0,0,1,1,1,1,1,0,0,1,0,0,3
platypus,1,0,1,1,0,1,1,0,1,1,0,0,4,1,0,1,1
polecat,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
pony,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,1,1
porpoise,0,0,0,1,0,1,1,1,1,1,0,1,0,1,0,1,1
puma,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
pussycat,1,0,0,1,0,0,1,1,1,1,0,0,4,1,1,1,1
raccoon,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
reindeer,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,1,1
rhea,0,1,1,0,0,0,1,0,1,1,0,0,2,1,0,1,2
scorpion,0,0,0,0,0,0,1,0,0,1,1,0,8,1,0,0,7
seahorse,0,0,1,0,0,1,0,1,1,0,0,1,0,1,0,0,4
seal,1,0,0,1,0,1,1,1,1,1,0,1,0,0,0,1,1
sealion,1,0,0,1,0,1,1,1,1,1,0,1,2,1,0,1,1
seasnake,0,0,0,0,0,1,1,1,1,0,1,0,0,1,0,0,3
seawasp,0,0,1,0,0,1,1,0,0,0,1,0,0,0,0,0,7
skimmer,0,1,1,0,1,1,1,0,1,1,0,0,2,1,0,0,2
skua,0,1,1,0,1,1,1,0,1,1,0,0,2,1,0,0,2
slowworm,0,0,1,0,0,0,1,1,1,1,0,0,0,1,0,0,3
slug,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,7
sole,0,0,1,0,0,1,0,1,1,0,0,1,0,1,0,0,4
sparrow,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,0,2
squirrel,1,0,0,1,0,0,0,1,1,1,0,0,2,1,0,0,1
starfish,0,0,1,0,0,1,1,0,0,0,0,0,5,0,0,0,7
stingray,0,0,1,0,0,1,1,1,1,0,1,1,0,1,0,1,4
swan,0,1,1,0,1,1,0,0,1,1,0,0,2,1,0,1,2
termite,0,0,1,0,0,0,0,0,0,1,0,0,6,0,0,0,6
toad,0,0,1,0,0,1,0,1,1,1,0,0,4,0,0,0,5
tortoise,0,0,1,0,0,0,0,0,1,1,0,0,4,1,0,1,3
tuatara,0,0,1,0,0,0,1,1,1,1,0,0,4,1,0,0,3
tuna,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,1,4
vampire,1,0,0,1,1,0,0,1,1,1,0,0,2,1,0,0,1
vole,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,0,1
vulture,0,1,1,0,1,0,1,0,1,1,0,0,2,1,0,1,2
wallaby,1,0,0,1,0,0,0,1,1,1,0,0,2,1,0,1,1
wasp,1,0,1,0,1,0,0,0,0,1,1,0,6,0,0,0,6
wolf,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
worm,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,7
wren,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,0,2"""
    Path("zoo.csv").write_text(zoo + "\n", encoding="utf-8")

    cls = """Class_Number,Number_Of_Animal_Species_In_Class,Class_Type,Animal_Names
1,41,Mammal,"aardvark, antelope, bear, boar, buffalo, calf, cavy, cheetah, deer, dolphin, elephant, fruitbat, giraffe, girl, goat, gorilla, hamster, hare, leopard, lion, lynx, mink, mole, mongoose, opossum, oryx, platypus, polecat, pony, porpoise, puma, pussycat, raccoon, reindeer, seal, sealion, squirrel, vampire, vole, wallaby, wolf"
2,20,Bird,"chicken, crow, dove, duck, flamingo, gull, hawk, kiwi, lark, ostrich, parakeet, penguin, pheasant, rhea, skimmer, skua, sparrow, swan, vulture, wren"
3,5,Reptile,"pitviper, seasnake, slowworm, tortoise, tuatara"
4,13,Fish,"bass, carp, catfish, chub, dogfish, haddock, herring, pike, piranha, seahorse, sole, stingray, tuna"
5,4,Amphibian,"frog, frog, newt, toad"
6,8,Bug,"flea, gnat, honeybee, housefly, ladybird, moth, termite, wasp"
7,10,Invertebrate,"clam, crab, crayfish, lobster, octopus, scorpion, seawasp, slug, starfish, worm"
"""
    Path("class.csv").write_text(cls + "\n", encoding="utf-8")

    aux = """[
  {"animal_name":"aardvark","habitat":"savanna","diet":"insectivore","conservation_status":"least concern"},
  {"animal_name":"antelope","habitat":"grasslands","diet":"herbivore","conservation":"near threatened"},
  {"animal_name":"bass","habitat":"freshwater","diet":"carnivore","conservation_status":"least"},
  {"animal_name":"bear","habitats":"forest","diet":"omnivore","conservation_status":"vulnerable"},
  {"animal_name":"boar","habitat":"forest","diet":"omnivor","status":"least concern"},
  {"animal_name":"buffalo","habitat":"grasslands","diet_type":"herbivore","conservation":"endangered"},
  {"animal_name":"calf","habitat":"domestic","diet":"herbivore"},
  {"animal_name":"carp","habitat":"fresh water","diet":"omnivore","conservation_status":"least concern"},
  {"animal_name":"catfish","habitat":"FreshWater","diet":"carnivore"},
  {"animal_name":"clam","habitat":"marine","diet":"filter_feeder","conservation_status":"least concern"},
  {"animal_name":"crab","habitat":"marine/coastal","diet":"omnivore","conservation":"least concern"},
  {"animal_name":"deer","habitat":"Forest","diet":"Herbivore","conservation_status":"vulnerable"}
]"""
    Path("auxiliary_metadata.json").write_text(aux + "\n", encoding="utf-8")


# ------------------- 2. LOAD AUX JSON -------------------
def load_aux_json(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        df = pd.DataFrame(data)
        df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
        return df
    except:
        return pd.DataFrame()


# ------------------- 3. MAIN FUNCTION -------------------
def gamma_load_and_integration():
    # Load zoo
    zoo = pd.read_csv("zoo.csv")
    zoo.columns = [c.strip().lower() for c in zoo.columns]
    zoo["animal_name_lower"] = zoo["animal_name"].str.lower()

    # Load class + explode
    cls = pd.read_csv("class.csv")
    cls.columns = [c.strip().lower() for c in cls.columns]
    cls_expl = (
        cls.assign(animal_name=cls["animal_names"].str.split(", "))
           .explode("animal_name")
           .drop(columns=["animal_names", "number_of_animal_species_in_class"])
    )
    cls_expl["animal_name"] = cls_expl["animal_name"].str.strip()
    cls_expl["animal_name_lower"] = cls_expl["animal_name"].str.lower()
    cls_clean = cls_expl[["animal_name_lower", "class_number", "class_type"]]

    # Load aux
    aux = load_aux_json("auxiliary_metadata.json")
    if not aux.empty:
        aux["animal_name_lower"] = aux["animal_name"].str.lower()

    # Merge
    df = zoo.merge(cls_clean, on="animal_name_lower", how="left")
    if not aux.empty:
        df = df.merge(aux, on="animal_name_lower", how="left")

    # --- NAME NORMALISATION ---
    df = df.sort_values("animal_name_lower") \
           .drop_duplicates(subset="animal_name_lower", keep="first")

    # Restore original zoo order
    df = df.set_index("animal_name_lower") \
           .reindex(zoo["animal_name_lower"]) \
           .reset_index(drop=True)

    # Final column order - preserve 'animal_name' and 'class_type'
    zoo_cols = [c for c in zoo.columns if c not in ["animal_name", "animal_name_lower"]]
    aux_cols = [c for c in aux.columns if c != "animal_name_lower"] if not aux.empty else []
    final_cols = ["animal_name"] + zoo_cols + ["class_number", "class_type"] + aux_cols
    df = df[final_cols]

    return df


# ------------------- 4. RUN -------------------
print("Creating files...")
create_files()

print("Running gamma_load_and_integration() with name normalisation...\n")
gamma_df = gamma_load_and_integration()

print(gamma_df.head())
print("\nShape after deduplication:", gamma_df.shape)
print("Columns:", gamma_df.columns.tolist())

Creating files...
Running gamma_load_and_integration() with name normalisation...

  animal_name  hair  feathers  eggs  milk  airborne aquatic  predator  \
0    aardvark     1         0     0     1         0       0         1   
1    antelope     1         0     0     1         0       0         0   
2        bass     0         0     1     0         0       1         1   
3        bear     1         0     0     1         0       0         1   
4        boar     1         0     0     1         0       0         1   

   toothed  backbone  ...  catsize  class_number  class_type     habitat  \
0        1         1  ...        1             1      Mammal     savanna   
1        1         1  ...        1             1      Mammal  grasslands   
2        1         1  ...        0             4        Fish  freshwater   
3        1         1  ...        1             1      Mammal         NaN   
4        1         1  ...        1             1      Mammal      forest   

          diet  conse

In [37]:
# ==============================================================
# TASK 1 + b + c :  Full integration + JSON cleanup (NO ERRORS)
# ==============================================================

import pandas as pd
import json
from pathlib import Path
import numpy as np

# ------------------- 1. CREATE 3 FILES -------------------
def create_files():
    zoo = """animal_name,hair,feathers,eggs,milk,airborne,aquatic,predator,toothed,backbone,breathes,venomous,fins,legs,tail,domestic,catsize,class_type
aardvark,1,0,0,1,0,0,1,1,1,1,0,0,4,0,0,1,1
antelope,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
bass,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
bear,1,0,0,1,0,0,1,1,1,1,0,0,4,0,0,1,1
boar,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
buffalo,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
calf,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,1,1
carp,0,0,1,0,0,1,0,1,1,0,0,1,0,1,1,0,4
catfish,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
cavy,1,0,0,1,0,0,0,1,1,1,0,0,4,0,1,0,1
cheetah,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
chicken,0,1,1,0,1,0,0,0,1,1,0,0,2,1,1,0,2
chub,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
clam,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,7
crab,0,0,1,0,0,1,1,0,0,0,0,0,4,0,0,0,7
crayfish,0,0,1,0,0,1,1,0,0,0,0,0,6,0,0,0,7
crow,0,1,1,0,1,0,1,0,1,1,0,0,2,1,0,0,2
deer,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
dogfish,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,1,4
dolphin,0,0,0,1,0,1,1,1,1,1,0,1,0,1,0,1,1
dove,0,1,1,0,1,0,0,0,1,1,0,0,2,1,1,0,2
duck,0,1,1,0,1,1,0,0,1,1,0,0,2,1,0,0,2
elephant,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
flamingo,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,1,2
flea,0,0,1,0,0,0,0,0,0,1,0,0,6,0,0,0,6
frog,0,0,1,0,0,1,1,1,1,1,0,0,4,0,0,0,5
frog,0,0,1,0,0,1,1,1,1,1,1,0,4,0,0,0,5
fruitbat,1,0,0,1,1,0,0,1,1,1,0,0,2,1,0,0,1
giraffe,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
girl,1,0,0,1,0,0,1,1,1,1,0,0,2,0,1,1,1
gnat,0,0,1,0,1,0,0,0,0,1,0,0,6,0,0,0,6
goat,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,1,1
gorilla,1,0,0,1,0,0,0,1,1,1,0,0,2,0,0,1,1
gull,0,1,1,0,1,1,1,0,1,1,0,0,2,1,0,0,2
haddock,0,0,1,0,0,1,0,1,1,0,0,1,0,1,0,0,4
hamster,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,0,1
hare,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,0,1
hawk,0,1,1,0,1,0,1,0,1,1,0,0,2,1,0,0,2
herring,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
honeybee,1,0,1,0,1,0,0,0,0,1,1,0,6,0,1,0,6
housefly,1,0,1,0,1,0,0,0,0,1,0,0,6,0,0,0,6
kiwi,0,1,1,0,0,0,1,0,1,1,0,0,2,1,0,0,2
ladybird,0,0,1,0,1,0,1,0,0,1,0,0,6,0,0,0,6
lark,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,0,2
leopard,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
lion,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
lobster,0,0,1,0,0,1,1,0,0,0,0,0,6,0,0,0,7
lynx,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
mink,1,0,0,1,0,1,1,1,1,1,0,0,4,1,0,1,1
mole,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,0,1
mongoose,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
moth,1,0,1,0,1,0,0,0,0,1,0,0,6,0,0,0,6
newt,0,0,1,0,0,1,1,1,1,1,0,0,4,1,0,0,5
octopus,0,0,1,0,0,1,1,0,0,0,0,0,8,0,0,1,7
opossum,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,0,1
oryx,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
ostrich,0,1,1,0,0,0,0,0,1,1,0,0,2,1,0,1,2
parakeet,0,1,1,0,1,0,0,0,1,1,0,0,2,1,1,0,2
penguin,0,1,1,0,0,1,1,0,1,1,0,0,2,1,0,1,2
pheasant,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,0,2
pike,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,1,4
piranha,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
pitviper,0,0,1,0,0,0,1,1,1,1,1,0,0,1,0,0,3
platypus,1,0,1,1,0,1,1,0,1,1,0,0,4,1,0,1,1
polecat,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
pony,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,1,1
porpoise,0,0,0,1,0,1,1,1,1,1,0,1,0,1,0,1,1
puma,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
pussycat,1,0,0,1,0,0,1,1,1,1,0,0,4,1,1,1,1
raccoon,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
reindeer,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,1,1
rhea,0,1,1,0,0,0,1,0,1,1,0,0,2,1,0,1,2
scorpion,0,0,0,0,0,0,1,0,0,1,1,0,8,1,0,0,7
seahorse,0,0,1,0,0,1,0,1,1,0,0,1,0,1,0,0,4
seal,1,0,0,1,0,1,1,1,1,1,0,1,0,0,0,1,1
sealion,1,0,0,1,0,1,1,1,1,1,0,1,2,1,0,1,1
seasnake,0,0,0,0,0,1,1,1,1,0,1,0,0,1,0,0,3
seawasp,0,0,1,0,0,1,1,0,0,0,1,0,0,0,0,0,7
skimmer,0,1,1,0,1,1,1,0,1,1,0,0,2,1,0,0,2
skua,0,1,1,0,1,1,1,0,1,1,0,0,2,1,0,0,2
slowworm,0,0,1,0,0,0,1,1,1,1,0,0,0,1,0,0,3
slug,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,7
sole,0,0,1,0,0,1,0,1,1,0,0,1,0,1,0,0,4
sparrow,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,0,2
squirrel,1,0,0,1,0,0,0,1,1,1,0,0,2,1,0,0,1
starfish,0,0,1,0,0,1,1,0,0,0,0,0,5,0,0,0,7
stingray,0,0,1,0,0,1,1,1,1,0,1,1,0,1,0,1,4
swan,0,1,1,0,1,1,0,0,1,1,0,0,2,1,0,1,2
termite,0,0,1,0,0,0,0,0,0,1,0,0,6,0,0,0,6
toad,0,0,1,0,0,1,0,1,1,1,0,0,4,0,0,0,5
tortoise,0,0,1,0,0,0,0,0,1,1,0,0,4,1,0,1,3
tuatara,0,0,1,0,0,0,1,1,1,1,0,0,4,1,0,0,3
tuna,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,1,4
vampire,1,0,0,1,1,0,0,1,1,1,0,0,2,1,0,0,1
vole,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,0,1
vulture,0,1,1,0,1,0,1,0,1,1,0,0,2,1,0,1,2
wallaby,1,0,0,1,0,0,0,1,1,1,0,0,2,1,0,1,1
wasp,1,0,1,0,1,0,0,0,0,1,1,0,6,0,0,0,6
wolf,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
worm,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,7
wren,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,0,2"""
    Path("zoo.csv").write_text(zoo + "\n", encoding="utf-8")

    cls = """Class_Number,Number_Of_Animal_Species_In_Class,Class_Type,Animal_Names
1,41,Mammal,"aardvark, antelope, bear, boar, buffalo, calf, cavy, cheetah, deer, dolphin, elephant, fruitbat, giraffe, girl, goat, gorilla, hamster, hare, leopard, lion, lynx, mink, mole, mongoose, opossum, oryx, platypus, polecat, pony, porpoise, puma, pussycat, raccoon, reindeer, seal, sealion, squirrel, vampire, vole, wallaby, wolf"
2,20,Bird,"chicken, crow, dove, duck, flamingo, gull, hawk, kiwi, lark, ostrich, parakeet, penguin, pheasant, rhea, skimmer, skua, sparrow, swan, vulture, wren"
3,5,Reptile,"pitviper, seasnake, slowworm, tortoise, tuatara"
4,13,Fish,"bass, carp, catfish, chub, dogfish, haddock, herring, pike, piranha, seahorse, sole, stingray, tuna"
5,4,Amphibian,"frog, frog, newt, toad"
6,8,Bug,"flea, gnat, honeybee, housefly, ladybird, moth, termite, wasp"
7,10,Invertebrate,"clam, crab, crayfish, lobster, octopus, scorpion, seawasp, slug, starfish, worm"
"""
    Path("class.csv").write_text(cls + "\n", encoding="utf-8")

    aux = """[
  {"animal_name":"aardvark","habitat":"savanna","diet":"insectivore","conservation_status":"least concern"},
  {"animal_name":"antelope","habitat":"grasslands","diet":"herbivore","conservation":"near threatened"},
  {"animal_name":"bass","habitat":"freshwater","diet":"carnivore","conservation_status":"least"},
  {"animal_name":"bear","habitats":"forest","diet":"omnivore","conservation_status":"vulnerable"},
  {"animal_name":"boar","habitat":"forest","diet":"omnivor","status":"least concern"},
  {"animal_name":"buffalo","habitat":"grasslands","diet_type":"herbivore","conservation":"endangered"},
  {"animal_name":"calf","habitat":"domestic","diet":"herbivore"},
  {"animal_name":"carp","habitat":"fresh water","diet":"omnivore","conservation_status":"least concern"},
  {"animal_name":"catfish","habitat":"FreshWater","diet":"carnivore"},
  {"animal_name":"clam","habitat":"marine","diet":"filter_feeder","conservation_status":"least concern"},
  {"animal_name":"crab","habitat":"marine/coastal","diet":"omnivore","conservation":"least concern"},
  {"animal_name":"deer","habitat":"Forest","diet":"Herbivore","conservation_status":"vulnerable"}
]"""
    Path("auxiliary_metadata.json").write_text(aux + "\n", encoding="utf-8")


# ------------------- 2. CLEAN & STANDARDISE AUX JSON -------------------
def clean_aux_metadata(df):
    if df.empty:
        return df

    # 1. Standardise column names
    rename_map = {}
    for col in df.columns:
        low = col.lower().strip()
        if low in ["conservation_status", "conservation", "status"]:
            rename_map[col] = "conservation_status"
        elif low in ["habitat", "habitats"]:
            rename_map[col] = "habitat_type"
        elif low in ["diet", "diet_type"]:
            rename_map[col] = "diet"
        else:
            rename_map[col] = low.replace(" ", "_")
    df = df.rename(columns=rename_map)

    # 2. Safe string cleaning (only if column exists & is object)
    def safe_clean(series):
        if series.name in df.columns and pd.api.types.is_object_dtype(series):
            return series.astype(str).str.lower().str.strip()
        return series

    # diet
    if "diet" in df.columns:
        d = safe_clean(df["diet"])
        d = d.replace({"omnivor":"omnivore","herbivor":"herbivore","carnivor":"carnivore"})
        df["diet"] = d.str.title()

    # habitat_type
    if "habitat_type" in df.columns:
        h = safe_clean(df["habitat_type"])
        h = h.replace({"fresh water":"freshwater","marine/coastal":"marine_coastal"})
        h = h.str.replace(r"\s+"," ",regex=True).str.title()
        df["habitat_type"] = h

    # merge key
    if "animal_name" in df.columns:
        df["animal_name_lower"] = safe_clean(df["animal_name"])

    return df


# ------------------- 3. MAIN INTEGRATION FUNCTION -------------------
def gamma_load_and_integration():
    # ---- zoo -------------------------------------------------
    zoo = pd.read_csv("zoo.csv")
    zoo.columns = [c.strip().lower() for c in zoo.columns]
    zoo["animal_name_lower"] = zoo["animal_name"].astype(str).str.lower().str.strip()
    zoo_order = zoo["animal_name_lower"].copy()

    # ---- class + explode ------------------------------------
    cls = pd.read_csv("class.csv")
    cls.columns = [c.strip().lower() for c in cls.columns]
    cls_expl = (
        cls.assign(animal_name=cls["animal_names"].str.split(", "))
           .explode("animal_name")
           .drop(columns=["animal_names","number_of_animal_species_in_class"])
    )
    cls_expl["animal_name"] = cls_expl["animal_name"].str.strip()
    cls_expl["animal_name_lower"] = cls_expl["animal_name"].str.lower()
    cls_clean = cls_expl[["animal_name_lower","class_number","class_type"]]

    # ---- auxiliary -------------------------------------------
    try:
        with open("auxiliary_metadata.json","r",encoding="utf-8") as f:
            data = json.load(f)
        aux_raw = pd.DataFrame(data)
    except Exception as e:
        print("Warning: auxiliary file not loaded:", e)
        aux_raw = pd.DataFrame()
    aux = clean_aux_metadata(aux_raw)

    # ---- merge -----------------------------------------------
    df = zoo.merge(cls_clean, on="animal_name_lower", how="left")
    if not aux.empty and "animal_name_lower" in aux.columns:
        df = df.merge(aux.drop(columns=["animal_name"],errors="ignore"),
                      on="animal_name_lower", how="left")

    # ---- name normalisation (dedupe case‑insensitive) -------
    df = df.sort_values("animal_name_lower")\
           .drop_duplicates(subset="animal_name_lower", keep="first")

    # ---- restore original zoo order -------------------------
    df = df.set_index("animal_name_lower").reindex(zoo_order).reset_index(drop=True)

    # ---- final column order ---------------------------------
    zoo_cols = [c for c in zoo.columns if c not in ["animal_name","animal_name_lower"]]
    aux_cols = [c for c in aux.columns if c not in ["animal_name","animal_name_lower"]]
    final_cols = ["animal_name"] + zoo_cols + ["class_number","class_type"] + aux_cols
    df = df[final_cols]

    return df


# ------------------- 4. RUN -------------------
print("Creating files...")
create_files()

print("Running gamma_load_and_integration() with JSON cleanup...\n")
gamma_df = gamma_load_and_integration()

print(gamma_df.head(10))
print("\nShape:", gamma_df.shape)
print("Columns:", gamma_df.columns.tolist())

Creating files...
Running gamma_load_and_integration() with JSON cleanup...

  animal_name  hair  feathers  eggs  milk  airborne  aquatic  predator  \
0    aardvark     1         0     0     1         0        0         1   
1    antelope     1         0     0     1         0        0         0   
2        bass     0         0     1     0         0        1         1   
3        bear     1         0     0     1         0        0         1   
4        boar     1         0     0     1         0        0         1   
5     buffalo     1         0     0     1         0        0         0   
6        calf     1         0     0     1         0        0         0   
7        carp     0         0     1     0         0        1         0   
8     catfish     0         0     1     0         0        1         1   
9        cavy     1         0     0     1         0        0         0   

   toothed  backbone  ...  venomous  fins  legs  tail  domestic  catsize  \
0        1         1  ...       

In [50]:
# --------------------------------------------------------------
#  TASK F – FEATURE ENGINEERING (conservation_priority + aquatic_flag)
# --------------------------------------------------------------

import pandas as pd
import json
from pathlib import Path
from typing import List, Dict, Any

# --------------------------------------------------------------
# 1. CREATE FILES
# --------------------------------------------------------------
def _create_files():
    zoo = """animal_name,hair,feathers,eggs,milk,airborne,aquatic,predator,toothed,backbone,breathes,venomous,fins,legs,tail,domestic,catsize,class_type
aardvark,1,0,0,1,0,0,1,1,1,1,0,0,4,0,0,1,1
antelope,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
bass,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
bear,1,0,0,1,0,0,1,1,1,1,0,0,4,0,0,1,1
boar,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
buffalo,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
calf,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,1,1
carp,0,0,1,0,0,1,0,1,1,0,0,1,0,1,1,0,4
catfish,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
cavy,1,0,0,1,0,0,0,1,1,1,0,0,4,0,1,0,1
cheetah,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
chicken,0,1,1,0,1,0,0,0,1,1,0,0,2,1,1,0,2
chub,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
clam,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,7
crab,0,0,1,0,0,1,1,0,0,0,0,0,4,0,0,0,7
crayfish,0,0,1,0,0,1,1,0,0,0,0,0,6,0,0,0,7
crow,0,1,1,0,1,0,1,0,1,1,0,0,2,1,0,0,2
deer,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
dogfish,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,1,4
dolphin,0,0,0,1,0,1,1,1,1,1,0,1,0,1,0,1,1
dove,0,1,1,0,1,0,0,0,1,1,0,0,2,1,1,0,2
duck,0,1,1,0,1,1,0,0,1,1,0,0,2,1,0,0,2
elephant,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
flamingo,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,1,2
flea,0,0,1,0,0,0,0,0,0,1,0,0,6,0,0,0,6
frog,0,0,1,0,0,1,1,1,1,1,0,0,4,0,0,0,5
frog,0,0,1,0,0,1,1,1,1,1,1,0,4,0,0,0,5
fruitbat,1,0,0,1,1,0,0,1,1,1,0,0,2,1,0,0,1
giraffe,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
girl,1,0,0,1,0,0,1,1,1,1,0,0,2,0,1,1,1
gnat,0,0,1,0,1,0,0,0,0,1,0,0,6,0,0,0,6
goat,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,1,1
gorilla,1,0,0,1,0,0,0,1,1,1,0,0,2,0,0,1,1
gull,0,1,1,0,1,1,1,0,1,1,0,0,2,1,0,0,2
haddock,0,0,1,0,0,1,0,1,1,0,0,1,0,1,0,0,4
hamster,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,0,1
hare,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,0,1
hawk,0,1,1,0,1,0,1,0,1,1,0,0,2,1,0,0,2
herring,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
honeybee,1,0,1,0,1,0,0,0,0,1,1,0,6,0,1,0,6
housefly,1,0,1,0,1,0,0,0,0,1,0,0,6,0,0,0,6
kiwi,0,1,1,0,0,0,1,0,1,1,0,0,2,1,0,0,2
ladybird,0,0,1,0,1,0,1,0,0,1,0,0,6,0,0,0,6
lark,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,0,2
leopard,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
lion,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
lobster,0,0,1,0,0,1,1,0,0,0,0,0,6,0,0,0,7
lynx,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
mink,1,0,0,1,0,1,1,1,1,1,0,0,4,1,0,1,1
mole,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,0,1
mongoose,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
moth,1,0,1,0,1,0,0,0,0,1,0,0,6,0,0,0,6
newt,0,0,1,0,0,1,1,1,1,1,0,0,4,1,0,0,5
octopus,0,0,1,0,0,1,1,0,0,0,0,0,8,0,0,1,7
opossum,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,0,1
oryx,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
ostrich,0,1,1,0,0,0,0,0,1,1,0,0,2,1,0,1,2
parakeet,0,1,1,0,1,0,0,0,1,1,0,0,2,1,1,0,2
penguin,0,1,1,0,0,1,1,0,1,1,0,0,2,1,0,1,2
pheasant,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,0,2
pike,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,1,4
piranha,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
pitviper,0,0,1,0,0,0,1,1,1,1,1,0,0,1,0,0,3
platypus,1,0,1,1,0,1,1,0,1,1,0,0,4,1,0,1,1
polecat,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
pony,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,1,1
porpoise,0,0,0,1,0,1,1,1,1,1,0,1,0,1,0,1,1
puma,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
pussycat,1,0,0,1,0,0,1,1,1,1,0,0,4,1,1,1,1
raccoon,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
reindeer,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,1,1
rhea,0,1,1,0,0,0,1,0,1,1,0,0,2,1,0,1,2
scorpion,0,0,0,0,0,0,1,0,0,1,1,0,8,1,0,0,7
seahorse,0,0,1,0,0,1,0,1,1,0,0,1,0,1,0,0,4
seal,1,0,0,1,0,1,1,1,1,1,0,1,0,0,0,1,1
sealion,1,0,0,1,0,1,1,1,1,1,0,1,2,1,0,1,1
seasnake,0,0,0,0,0,1,1,1,1,0,1,0,0,1,0,0,3
seawasp,0,0,1,0,0,1,1,0,0,0,1,0,0,0,0,0,7
skimmer,0,1,1,0,1,1,1,0,1,1,0,0,2,1,0,0,2
skua,0,1,1,0,1,1,1,0,1,1,0,0,2,1,0,0,2
slowworm,0,0,1,0,0,0,1,1,1,1,0,0,0,1,0,0,3
slug,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,7
sole,0,0,1,0,0,1,0,1,1,0,0,1,0,1,0,0,4
sparrow,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,0,2
squirrel,1,0,0,1,0,0,0,1,1,1,0,0,2,1,0,0,1
starfish,0,0,1,0,0,1,1,0,0,0,0,0,5,0,0,0,7
stingray,0,0,1,0,0,1,1,1,1,0,1,1,0,1,0,1,4
swan,0,1,1,0,1,1,0,0,1,1,0,0,2,1,0,1,2
termite,0,0,1,0,0,0,0,0,0,1,0,0,6,0,0,0,6
toad,0,0,1,0,0,1,0,1,1,1,0,0,4,0,0,0,5
tortoise,0,0,1,0,0,0,0,0,1,1,0,0,4,1,0,1,3
tuatara,0,0,1,0,0,0,1,1,1,1,0,0,4,1,0,0,3
tuna,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,1,4
vampire,1,0,0,1,1,0,0,1,1,1,0,0,2,1,0,0,1
vole,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,0,1
vulture,0,1,1,0,1,0,1,0,1,1,0,0,2,1,0,1,2
wallaby,1,0,0,1,0,0,0,1,1,1,0,0,2,1,0,1,1
wasp,1,0,1,0,1,0,0,0,0,1,1,0,6,0,0,0,6
wolf,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
worm,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,7
wren,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,0,2"""
    Path("/content/zoo.csv").write_text(zoo.strip() + "\n")

    cls = """Class_Number,Number_Of_Animal_Species_In_Class,Class_Type,Animal_Names
1,41,Mammal,"aardvark, antelope, bear, boar, buffalo, calf, cavy, cheetah, deer, dolphin, elephant, fruitbat, giraffe, girl, goat, gorilla, hamster, hare, leopard, lion, lynx, mink, mole, mongoose, opossum, oryx, platypus, polecat, pony, porpoise, puma, pussycat, raccoon, reindeer, seal, sealion, squirrel, vampire, vole, wallaby, wolf"
2,20,Bird,"chicken, crow, dove, duck, flamingo, gull, hawk, kiwi, lark, ostrich, parakeet, penguin, pheasant, rhea, skimmer, skua, sparrow, swan, vulture, wren"
3,5,Reptile,"pitviper, seasnake, slowworm, tortoise, tuatara"
4,13,Fish,"bass, carp, catfish, chub, dogfish, haddock, herring, pike, piranha, seahorse, sole, stingray, tuna"
5,4,Amphibian,"frog, frog, newt, toad"
6,8,Bug,"flea, gnat, honeybee, housefly, ladybird, moth, termite, wasp"
7,10,Invertebrate,"clam, crab, crayfish, lobster, octopus, scorpion, seawasp, slug, starfish, worm"
"""
    Path("/content/class.csv").write_text(cls.strip() + "\n")

    aux = """[
  {"animal_name":"aardvark","habitat":"savanna","diet":"insectivore","conservation_status":"least concern"},
  {"animal_name":"antelope","habitat":"grasslands","diet":"herbivore","conservation":"near threatened"},
  {"animal_name":"bass","habitat":"freshwater","diet":"carnivore","conservation_status":"least"},
  {"animal_name":"bear","habitats":"forest","diet":"omnivore","conservation_status":"vulnerable"},
  {"animal_name":"boar","habitat":"forest","diet":"omnivor","status":"least concern"},
  {"animal_name":"buffalo","habitat":"grasslands","diet_type":"herbivore","conservation":"endangered"},
  {"animal_name":"calf","habitat":"domestic","diet":"herbivore"},
  {"animal_name":"carp","habitat":"fresh water","diet":"omnivore","conservation_status":"least concern"},
  {"animal_name":"catfish","habitat":"FreshWater","diet":"carnivore"},
  {"animal_name":"clam","habitat":"marine","diet":"filter_feeder","conservation_status":"least concern"},
  {"animal_name":"crab","habitat":"marine/coastal","diet":"omnivore","conservation":"least concern"},
  {"animal_name":"deer","habitat":"Forest","diet":"Herbivore","conservation_status":"vulnerable"}
]"""
    Path("/content/auxiliary_metadata.json").write_text(aux.strip() + "\n")


# --------------------------------------------------------------
# 2. ROBUST JSON LOADER
# --------------------------------------------------------------
def _load_aux_json(p: Path) -> List[Dict[str, Any]]:
    out = []
    with p.open(encoding="utf-8") as f:
        try:
            data = json.load(f)
        except:
            f.seek(0)
            data = [json.loads(l) for l in f if l.strip()]
        if not isinstance(data, list):
            return out
        for rec in data:
            try:
                norm = {k.strip().lower().replace(" ", "_"): v for k, v in rec.items()}
                if "animal_name" not in norm:
                    continue
                for k in {"habitat", "diet", "conservation_status", "conservation"}:
                    norm.setdefault(k, None)
                out.append(norm)
            except:
                continue
    return out


# --------------------------------------------------------------
# 3. MAIN FUNCTION + TASK E + TASK F
# --------------------------------------------------------------
def gamma_load_and_integration(
    zoo_path: str = "/content/zoo.csv",
    class_path: str = "/content/class.csv",
    aux_path: str = "/content/auxiliary_metadata.json"
) -> pd.DataFrame:

    # --- Load zoo ---
    zoo_df = pd.read_csv(zoo_path)
    zoo_df.columns = [c.strip().lower() for c in zoo_df.columns]

    # --- Load class & explode ---
    cls_df = pd.read_csv(class_path)
    cls_df.columns = [c.strip().lower() for c in cls_df.columns]

    cls_expl = (
        cls_df
        .assign(animal_name=cls_df["animal_names"].str.split(", "))
        .explode("animal_name")
        .drop(columns=["animal_names", "number_of_animal_species_in_class"], errors="ignore")
    )
    cls_expl["animal_name"] = cls_expl["animal_name"].astype(str).str.strip().str.lower()
    cls_clean = cls_expl[["animal_name", "class_number", "class_type"]]

    # --- Load aux ---
    aux_records = _load_aux_json(Path(aux_path))
    aux_df = pd.DataFrame(aux_records)
    if not aux_df.empty:
        aux_df["animal_name"] = aux_df["animal_name"].astype(str).str.strip().str.lower()

    # --- Merge ---
    df = zoo_df.copy()
    df["animal_name"] = df["animal_name"].astype(str).str.strip().str.lower()
    df = df.merge(cls_clean, on="animal_name", how="left")
    if not aux_df.empty:
        df = df.merge(aux_df, on="animal_name", how="left")

    # --- TASK E: Drop rows with missing aux data ---
    aux_cols = ["habitat", "diet", "conservation_status", "conservation"]
    aux_present = [c for c in aux_cols if c in df.columns]
    if aux_present:
        before = len(df)
        df = df.dropna(subset=aux_present, how="any")
        print(f"Task E: Dropped {before - len(df)} rows with missing aux data → {len(df)} remain.")

    # --- TASK F: Feature Engineering ---
    print("Task F: Adding 'conservation_priority' and 'aquatic_flag'...")

    # 1. conservation_priority
    priority_map = {
        "endangered": 5,
        "vulnerable": 4,
        "near threatened": 3,
        "least": 1,
        "least concern": 1,
    }

    def get_priority(val):
        if pd.isna(val):
            return 0
        val = str(val).lower().strip()
        for key, score in priority_map.items():
            if key in val:
                return score
        return 0  # unknown

    df["conservation_priority"] = df["conservation_status"].fillna(df["conservation"]).apply(get_priority)

    # 2. aquatic_flag
    df["aquatic_flag"] = df["habitat"].astype(str).str.lower().apply(
        lambda x: 1 if ("water" in x or "marine" in x) else 0
    )

    # --- Final formatting ---
    df["animal_name"] = df["animal_name"].str.title()
    zoo_cols = [c for c in zoo_df.columns if c != "animal_name"]
    final_cols = ["animal_name"] + zoo_cols + ["class_number", "class_type"] + aux_present + ["conservation_priority", "aquatic_flag"]
    df = df[[c for c in final_cols if c in df.columns]]

    return df


# --------------------------------------------------------------
# 4. RUN
# --------------------------------------------------------------
print("Creating files in /content ...")
_create_files()

print("\nRunning gamma_load_and_integration() + Task E + Task F...\n")
final_df = gamma_load_and_integration()

print("\nFirst 10 rows with new features:\n")
print(final_df[["animal_name", "habitat", "conservation_status", "conservation", "conservation_priority", "aquatic_flag"]].head(10))
print(f"\nFinal shape: {final_df.shape}")
print(f"Columns: {final_df.columns.tolist()}")

Creating files in /content ...

Running gamma_load_and_integration() + Task E + Task F...

Task E: Dropped 103 rows with missing aux data → 0 remain.
Task F: Adding 'conservation_priority' and 'aquatic_flag'...

First 10 rows with new features:

Empty DataFrame
Columns: [animal_name, habitat, conservation_status, conservation, conservation_priority, aquatic_flag]
Index: []

Final shape: (0, 24)
Columns: ['animal_name', 'hair', 'feathers', 'eggs', 'milk', 'airborne', 'aquatic', 'predator', 'toothed', 'backbone', 'breathes', 'venomous', 'fins', 'legs', 'tail', 'domestic', 'catsize', 'class_number', 'habitat', 'diet', 'conservation_status', 'conservation', 'conservation_priority', 'aquatic_flag']


In [47]:
# --------------------------------------------------------------
#  TASK 1 – gamma_load_and_integration() – FIXED & ROBUST
# --------------------------------------------------------------

import pandas as pd
import json
from pathlib import Path
from typing import List, Dict, Any

def _create_files():
    zoo = """animal_name,hair,feathers,eggs,milk,airborne,aquatic,predator,toothed,backbone,breathes,venomous,fins,legs,tail,domestic,catsize,class_type
aardvark,1,0,0,1,0,0,1,1,1,1,0,0,4,0,0,1,1
antelope,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
bass,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
bear,1,0,0,1,0,0,1,1,1,1,0,0,4,0,0,1,1
boar,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
buffalo,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
calf,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,1,1
carp,0,0,1,0,0,1,0,1,1,0,0,1,0,1,1,0,4
catfish,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
cavy,1,0,0,1,0,0,0,1,1,1,0,0,4,0,1,0,1
cheetah,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
chicken,0,1,1,0,1,0,0,0,1,1,0,0,2,1,1,0,2
chub,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
clam,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,7
crab,0,0,1,0,0,1,1,0,0,0,0,0,4,0,0,0,7
crayfish,0,0,1,0,0,1,1,0,0,0,0,0,6,0,0,0,7
crow,0,1,1,0,1,0,1,0,1,1,0,0,2,1,0,0,2
deer,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
dogfish,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,1,4
dolphin,0,0,0,1,0,1,1,1,1,1,0,1,0,1,0,1,1
dove,0,1,1,0,1,0,0,0,1,1,0,0,2,1,1,0,2
duck,0,1,1,0,1,1,0,0,1,1,0,0,2,1,0,0,2
elephant,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
flamingo,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,1,2
flea,0,0,1,0,0,0,0,0,0,1,0,0,6,0,0,0,6
frog,0,0,1,0,0,1,1,1,1,1,0,0,4,0,0,0,5
frog,0,0,1,0,0,1,1,1,1,1,1,0,4,0,0,0,5
fruitbat,1,0,0,1,1,0,0,1,1,1,0,0,2,1,0,0,1
giraffe,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
girl,1,0,0,1,0,0,1,1,1,1,0,0,2,0,1,1,1
gnat,0,0,1,0,1,0,0,0,0,1,0,0,6,0,0,0,6
goat,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,1,1
gorilla,1,0,0,1,0,0,0,1,1,1,0,0,2,0,0,1,1
gull,0,1,1,0,1,1,1,0,1,1,0,0,2,1,0,0,2
haddock,0,0,1,0,0,1,0,1,1,0,0,1,0,1,0,0,4
hamster,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,0,1
hare,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,0,1
hawk,0,1,1,0,1,0,1,0,1,1,0,0,2,1,0,0,2
herring,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
honeybee,1,0,1,0,1,0,0,0,0,1,1,0,6,0,1,0,6
housefly,1,0,1,0,1,0,0,0,0,1,0,0,6,0,0,0,6
kiwi,0,1,1,0,0,0,1,0,1,1,0,0,2,1,0,0,2
ladybird,0,0,1,0,1,0,1,0,0,1,0,0,6,0,0,0,6
lark,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,0,2
leopard,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
lion,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
lobster,0,0,1,0,0,1,1,0,0,0,0,0,6,0,0,0,7
lynx,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
mink,1,0,0,1,0,1,1,1,1,1,0,0,4,1,0,1,1
mole,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,0,1
mongoose,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
moth,1,0,1,0,1,0,0,0,0,1,0,0,6,0,0,0,6
newt,0,0,1,0,0,1,1,1,1,1,0,0,4,1,0,0,5
octopus,0,0,1,0,0,1,1,0,0,0,0,0,8,0,0,1,7
opossum,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,0,1
oryx,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
ostrich,0,1,1,0,0,0,0,0,1,1,0,0,2,1,0,1,2
parakeet,0,1,1,0,1,0,0,0,1,1,0,0,2,1,1,0,2
penguin,0,1,1,0,0,1,1,0,1,1,0,0,2,1,0,1,2
pheasant,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,0,2
pike,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,1,4
piranha,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
pitviper,0,0,1,0,0,0,1,1,1,1,1,0,0,1,0,0,3
platypus,1,0,1,1,0,1,1,0,1,1,0,0,4,1,0,1,1
polecat,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
pony,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,1,1
porpoise,0,0,0,1,0,1,1,1,1,1,0,1,0,1,0,1,1
puma,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
pussycat,1,0,0,1,0,0,1,1,1,1,0,0,4,1,1,1,1
raccoon,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
reindeer,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,1,1
rhea,0,1,1,0,0,0,1,0,1,1,0,0,2,1,0,1,2
scorpion,0,0,0,0,0,0,1,0,0,1,1,0,8,1,0,0,7
seahorse,0,0,1,0,0,1,0,1,1,0,0,1,0,1,0,0,4
seal,1,0,0,1,0,1,1,1,1,1,0,1,0,0,0,1,1
sealion,1,0,0,1,0,1,1,1,1,1,0,1,2,1,0,1,1
seasnake,0,0,0,0,0,1,1,1,1,0,1,0,0,1,0,0,3
seawasp,0,0,1,0,0,1,1,0,0,0,1,0,0,0,0,0,7
skimmer,0,1,1,0,1,1,1,0,1,1,0,0,2,1,0,0,2
skua,0,1,1,0,1,1,1,0,1,1,0,0,2,1,0,0,2
slowworm,0,0,1,0,0,0,1,1,1,1,0,0,0,1,0,0,3
slug,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,7
sole,0,0,1,0,0,1,0,1,1,0,0,1,0,1,0,0,4
sparrow,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,0,2
squirrel,1,0,0,1,0,0,0,1,1,1,0,0,2,1,0,0,1
starfish,0,0,1,0,0,1,1,0,0,0,0,0,5,0,0,0,7
stingray,0,0,1,0,0,1,1,1,1,0,1,1,0,1,0,1,4
swan,0,1,1,0,1,1,0,0,1,1,0,0,2,1,0,1,2
termite,0,0,1,0,0,0,0,0,0,1,0,0,6,0,0,0,6
toad,0,0,1,0,0,1,0,1,1,1,0,0,4,0,0,0,5
tortoise,0,0,1,0,0,0,0,0,1,1,0,0,4,1,0,1,3
tuatara,0,0,1,0,0,0,1,1,1,1,0,0,4,1,0,0,3
tuna,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,1,4
vampire,1,0,0,1,1,0,0,1,1,1,0,0,2,1,0,0,1
vole,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,0,1
vulture,0,1,1,0,1,0,1,0,1,1,0,0,2,1,0,1,2
wallaby,1,0,0,1,0,0,0,1,1,1,0,0,2,1,0,1,1
wasp,1,0,1,0,1,0,0,0,0,1,1,0,6,0,0,0,6
wolf,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
worm,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,7
wren,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,0,2"""
    Path("/content/zoo.csv").write_text(zoo.strip() + "\n")

    cls = """Class_Number,Number_Of_Animal_Species_In_Class,Class_Type,Animal_Names
1,41,Mammal,"aardvark, antelope, bear, boar, buffalo, calf, cavy, cheetah, deer, dolphin, elephant, fruitbat, giraffe, girl, goat, gorilla, hamster, hare, leopard, lion, lynx, mink, mole, mongoose, opossum, oryx, platypus, polecat, pony, porpoise, puma, pussycat, raccoon, reindeer, seal, sealion, squirrel, vampire, vole, wallaby, wolf"
2,20,Bird,"chicken, crow, dove, duck, flamingo, gull, hawk, kiwi, lark, ostrich, parakeet, penguin, pheasant, rhea, skimmer, skua, sparrow, swan, vulture, wren"
3,5,Reptile,"pitviper, seasnake, slowworm, tortoise, tuatara"
4,13,Fish,"bass, carp, catfish, chub, dogfish, haddock, herring, pike, piranha, seahorse, sole, stingray, tuna"
5,4,Amphibian,"frog, frog, newt, toad"
6,8,Bug,"flea, gnat, honeybee, housefly, ladybird, moth, termite, wasp"
7,10,Invertebrate,"clam, crab, crayfish, lobster, octopus, scorpion, seawasp, slug, starfish, worm"
"""
    Path("/content/class.csv").write_text(cls.strip() + "\n")

    aux = """[
  {"animal_name":"aardvark","habitat":"savanna","diet":"insectivore","conservation_status":"least concern"},
  {"animal_name":"antelope","habitat":"grasslands","diet":"herbivore","conservation":"near threatened"},
  {"animal_name":"bass","habitat":"freshwater","diet":"carnivore","conservation_status":"least"},
  {"animal_name":"bear","habitats":"forest","diet":"omnivore","conservation_status":"vulnerable"},
  {"animal_name":"boar","habitat":"forest","diet":"omnivor","status":"least concern"},
  {"animal_name":"buffalo","habitat":"grasslands","diet_type":"herbivore","conservation":"endangered"},
  {"animal_name":"calf","habitat":"domestic","diet":"herbivore"},
  {"animal_name":"carp","habitat":"fresh water","diet":"omnivore","conservation_status":"least concern"},
  {"animal_name":"catfish","habitat":"FreshWater","diet":"carnivore"},
  {"animal_name":"clam","habitat":"marine","diet":"filter_feeder","conservation_status":"least concern"},
  {"animal_name":"crab","habitat":"marine/coastal","diet":"omnivore","conservation":"least concern"},
  {"animal_name":"deer","habitat":"Forest","diet":"Herbivore","conservation_status":"vulnerable"}
]"""
    Path("/content/auxiliary_metadata.json").write_text(aux.strip() + "\n")


def _load_aux_json(p: Path) -> List[Dict[str, Any]]:
    out = []
    with p.open(encoding="utf-8") as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError:
            f.seek(0)
            data = [json.loads(l) for l in f if l.strip()]
        if not isinstance(data, list):
            return out
        for rec in data:
            try:
                norm = {k.strip().lower().replace(" ", "_"): v for k, v in rec.items()}
                if "animal_name" not in norm:
                    continue

                # Consolidate habitat keys
                habitat_keys = [k for k in norm.keys() if k in ['habitat', 'habitats']]
                if habitat_keys:
                    # Take the first one found as primary, concatenate others if present
                    norm['habitat_type'] = norm.pop(habitat_keys[0])
                    for k in habitat_keys[1:]:
                        if norm.get(k) is not None and norm.get(k) != '': # Check for non-empty values
                            norm['habitat_type'] = str(norm['habitat_type']) + '/' + str(norm.pop(k))
                else:
                    norm['habitat_type'] = None

                # Consolidate diet keys
                diet_keys = [k for k in norm.keys() if k in ['diet', 'diet_type']]
                if diet_keys:
                    norm['diet'] = norm.pop(diet_keys[0])
                    for k in diet_keys[1:]:
                        if norm.get(k) is not None and norm.get(k) != '':
                            norm['diet'] = str(norm['diet']) + '/' + str(norm.pop(k))
                else:
                    norm['diet'] = None

                # Consolidate conservation status keys
                conservation_keys = [k for k in norm.keys() if k in ['conservation_status', 'conservation', 'status']]
                if conservation_keys:
                    norm['conservation_status'] = norm.pop(conservation_keys[0])
                    for k in conservation_keys[1:]:
                        if norm.get(k) is not None and norm.get(k) != '':
                            norm['conservation_status'] = str(norm['conservation_status']) + '/' + str(norm.pop(k))
                else:
                    norm['conservation_status'] = None

                # Create a new dictionary for the final output, explicitly copying desired keys
                # This prevents unexpected keys from being passed through due to dynamic handling.
                norm_final = {
                    'animal_name': norm.get('animal_name'),
                    'habitat_type': norm.get('habitat_type'),
                    'diet': norm.get('diet'),
                    'conservation_status': norm.get('conservation_status')
                }
                # Copy any other generic keys not handled above that are still in 'norm'
                for k, v in norm.items():
                    if k not in norm_final: # Only add if not already explicitly handled
                        norm_final[k] = v

                out.append(norm_final)
            except Exception as e:
                print(f"Error processing auxiliary record: {rec} - {e}")
                continue
    return out


def gamma_load_and_integration(
    zoo_path: str = "/content/zoo.csv",
    class_path: str = "/content/class.csv",
    aux_path: str = "/content/auxiliary_metadata.json"
) -> pd.DataFrame:

    # ---- zoo ----------------------------------------------------
    zoo_df = pd.read_csv(zoo_path, encoding="utf-8")
    zoo_df.columns = [c.strip().lower() for c in zoo_df.columns]
    # Keep original animal_name for final output but create a lowercased version for merging
    zoo_df['original_animal_name'] = zoo_df['animal_name']
    zoo_df['animal_name'] = zoo_df['animal_name'].astype(str).str.strip().str.lower()

    # ---- class (explode animal_names) ---------------------------
    cls_df = pd.read_csv(class_path, encoding="utf-8")
    cls_df.columns = [c.strip().lower() for c in cls_df.columns]

    cls_expl = (
        cls_df
        .assign(animal_name=cls_df["animal_names"].str.split(", "))
        .explode("animal_name")
        .drop(columns=["animal_names", "number_of_animal_species_in_class"])
    )
    cls_expl["animal_name"] = cls_expl["animal_name"].astype(str).str.strip().str.lower()
    cls_clean = cls_expl[["animal_name", "class_number", "class_type"]]

    # ---- auxiliary JSON -----------------------------------------
    aux_records = _load_aux_json(Path(aux_path))
    aux_df = pd.DataFrame(aux_records)
    if not aux_df.empty:
        aux_df["animal_name"] = aux_df["animal_name"].astype(str).str.strip().str.lower()

        # Standardize string columns in aux_df if they exist
        for col_name in ['habitat_type', 'diet', 'conservation_status']:
            if col_name in aux_df.columns and pd.api.types.is_object_dtype(aux_df[col_name]):
                aux_df[col_name] = aux_df[col_name].astype(str).str.lower().str.strip()
                # Specific replacements for diet
                if col_name == 'diet':
                    aux_df[col_name] = aux_df[col_name].replace({"omnivor":"omnivore","herbivor":"herbivore","carnivor":"carnivore"}).str.title()
                # Specific replacements for habitat_type
                elif col_name == 'habitat_type':
                    aux_df[col_name] = aux_df[col_name].replace({"fresh water":"freshwater","marine/coastal":"marine_coastal"}).str.replace(r"\\s+"," ",regex=True).str.title()

    # ---- integration --------------------------------------------
    df = zoo_df.copy()

    # Merge with class data. This will create 'class_type_x' (from zoo_df's original 'class_type')
    # and 'class_type_y' (the descriptive class_type from cls_clean).
    df = df.merge(cls_clean, on="animal_name", how="left")

    # Merge with auxiliary data
    if not aux_df.empty:
        # Use suffixes to handle potential column name overlaps between main df and aux_df
        # (e.g., if aux also had 'hair', 'eggs' - though not in this specific aux_df, good practice)
        df = df.merge(aux_df, on="animal_name", how="left", suffixes=('', '_aux'))

    # --- Resolve conflicting 'class_type' columns (class_type_x from zoo, class_type_y from class.csv) ---
    # Ensure 'class_type' column is always created.
    df['class_type'] = pd.Series(dtype=object, index=df.index)

    # Prioritize descriptive class_type_y from class.csv
    if 'class_type_y' in df.columns:
        df['class_type'] = df['class_type_y']

    # Use numeric class_type_x from zoo.csv as a fallback for missing descriptive values
    if 'class_type_x' in df.columns:
        df['class_type'] = df['class_type'].fillna(df['class_type_x'])

    # Drop the original conflicting columns. 'errors=ignore' handles cases where they might not exist.
    df = df.drop(columns=['class_type_x', 'class_type_y'], errors='ignore')

    # ---- final tidy-up ------------------------------------------
    # Restore original animal_name casing, then drop the lowercased merge key
    df['animal_name'] = df['original_animal_name'].str.title()
    df = df.drop(columns=['original_animal_name'], errors='ignore')

    # Define final column order.
    # Start with the primary identifier.
    final_ordered_cols = ['animal_name']

    # Add remaining columns from the original zoo_df (excluding 'animal_name', 'original_animal_name',
    # and the original numeric 'class_type' which was resolved/dropped).
    for col in [c for c in zoo_df.columns if c not in ['animal_name', 'original_animal_name', 'class_type']]:
        if col in df.columns and col not in final_ordered_cols:
            final_ordered_cols.append(col)

    # Add 'class_number' and the descriptive 'class_type' (which was resolved from class_type_y)
    if 'class_number' in df.columns and 'class_type' in df.columns:
        if 'class_number' not in final_ordered_cols:
            final_ordered_cols.append('class_number')
        if 'class_type' not in final_ordered_cols:
            final_ordered_cols.append('class_type')

    # Add columns from auxiliary data that are still in the DataFrame.
    # Iterate through aux_df's columns (which are already normalized from _load_aux_json)
    if not aux_df.empty:
        for col in [c for c in aux_df.columns if c != 'animal_name']:
            if col in df.columns and col not in final_ordered_cols:
                final_ordered_cols.append(col)

    # Filter final_cols to include only columns actually present in df
    final_cols = [col for col in final_ordered_cols if col in df.columns]

    df = df[final_cols]

    return df


# RUN
print("Creating files in /content ...")
_create_files()

print("\nRunning gamma_load_and_integration() (Task 1)...")
gamma_df = gamma_load_and_integration()

print(gamma_df.head())
print("\nShape:", gamma_df.shape)
print("Columns:", gamma_df.columns.tolist())

Creating files in /content ...

Running gamma_load_and_integration() (Task 1)...
  animal_name  hair  feathers  eggs  milk  airborne  aquatic  predator  \
0    Aardvark     1         0     0     1         0        0         1   
1    Antelope     1         0     0     1         0        0         0   
2        Bass     0         0     1     0         0        1         1   
3        Bear     1         0     0     1         0        0         1   
4        Boar     1         0     0     1         0        0         1   

   toothed  backbone  ...  fins  legs  tail  domestic  catsize  class_number  \
0        1         1  ...     0     4     0         0        1             1   
1        1         1  ...     0     4     1         0        1             1   
2        1         1  ...     1     0     1         0        0             4   
3        1         1  ...     0     4     0         0        1             1   
4        1         1  ...     0     4     1         0        1            

In [49]:
# --------------------------------------------------------------
#  TASK 1 + TASK E – FULLY FIXED & DROP MISSING AUX DATA
# --------------------------------------------------------------

import pandas as pd
import json
from pathlib import Path
from typing import List, Dict, Any

# --------------------------------------------------------------
# 1. CREATE FILES
# --------------------------------------------------------------
def _create_files():
    zoo = """animal_name,hair,feathers,eggs,milk,airborne,aquatic,predator,toothed,backbone,breathes,venomous,fins,legs,tail,domestic,catsize,class_type
aardvark,1,0,0,1,0,0,1,1,1,1,0,0,4,0,0,1,1
antelope,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
bass,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
bear,1,0,0,1,0,0,1,1,1,1,0,0,4,0,0,1,1
boar,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
buffalo,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
calf,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,1,1
carp,0,0,1,0,0,1,0,1,1,0,0,1,0,1,1,0,4
catfish,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
cavy,1,0,0,1,0,0,0,1,1,1,0,0,4,0,1,0,1
cheetah,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
chicken,0,1,1,0,1,0,0,0,1,1,0,0,2,1,1,0,2
chub,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
clam,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,7
crab,0,0,1,0,0,1,1,0,0,0,0,0,4,0,0,0,7
crayfish,0,0,1,0,0,1,1,0,0,0,0,0,6,0,0,0,7
crow,0,1,1,0,1,0,1,0,1,1,0,0,2,1,0,0,2
deer,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
dogfish,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,1,4
dolphin,0,0,0,1,0,1,1,1,1,1,0,1,0,1,0,1,1
dove,0,1,1,0,1,0,0,0,1,1,0,0,2,1,1,0,2
duck,0,1,1,0,1,1,0,0,1,1,0,0,2,1,0,0,2
elephant,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
flamingo,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,1,2
flea,0,0,1,0,0,0,0,0,0,1,0,0,6,0,0,0,6
frog,0,0,1,0,0,1,1,1,1,1,0,0,4,0,0,0,5
frog,0,0,1,0,0,1,1,1,1,1,1,0,4,0,0,0,5
fruitbat,1,0,0,1,1,0,0,1,1,1,0,0,2,1,0,0,1
giraffe,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
girl,1,0,0,1,0,0,1,1,1,1,0,0,2,0,1,1,1
gnat,0,0,1,0,1,0,0,0,0,1,0,0,6,0,0,0,6
goat,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,1,1
gorilla,1,0,0,1,0,0,0,1,1,1,0,0,2,0,0,1,1
gull,0,1,1,0,1,1,1,0,1,1,0,0,2,1,0,0,2
haddock,0,0,1,0,0,1,0,1,1,0,0,1,0,1,0,0,4
hamster,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,0,1
hare,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,0,1
hawk,0,1,1,0,1,0,1,0,1,1,0,0,2,1,0,0,2
herring,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
honeybee,1,0,1,0,1,0,0,0,0,1,1,0,6,0,1,0,6
housefly,1,0,1,0,1,0,0,0,0,1,0,0,6,0,0,0,6
kiwi,0,1,1,0,0,0,1,0,1,1,0,0,2,1,0,0,2
ladybird,0,0,1,0,1,0,1,0,0,1,0,0,6,0,0,0,6
lark,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,0,2
leopard,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
lion,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
lobster,0,0,1,0,0,1,1,0,0,0,0,0,6,0,0,0,7
lynx,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
mink,1,0,0,1,0,1,1,1,1,1,0,0,4,1,0,1,1
mole,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,0,1
mongoose,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
moth,1,0,1,0,1,0,0,0,0,1,0,0,6,0,0,0,6
newt,0,0,1,0,0,1,1,1,1,1,0,0,4,1,0,0,5
octopus,0,0,1,0,0,1,1,0,0,0,0,0,8,0,0,1,7
opossum,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,0,1
oryx,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
ostrich,0,1,1,0,0,0,0,0,1,1,0,0,2,1,0,1,2
parakeet,0,1,1,0,1,0,0,0,1,1,0,0,2,1,1,0,2
penguin,0,1,1,0,0,1,1,0,1,1,0,0,2,1,0,1,2
pheasant,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,0,2
pike,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,1,4
piranha,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
pitviper,0,0,1,0,0,0,1,1,1,1,1,0,0,1,0,0,3
platypus,1,0,1,1,0,1,1,0,1,1,0,0,4,1,0,1,1
polecat,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
pony,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,1,1
porpoise,0,0,0,1,0,1,1,1,1,1,0,1,0,1,0,1,1
puma,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
pussycat,1,0,0,1,0,0,1,1,1,1,0,0,4,1,1,1,1
raccoon,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
reindeer,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,1,1
rhea,0,1,1,0,0,0,1,0,1,1,0,0,2,1,0,1,2
scorpion,0,0,0,0,0,0,1,0,0,1,1,0,8,1,0,0,7
seahorse,0,0,1,0,0,1,0,1,1,0,0,1,0,1,0,0,4
seal,1,0,0,1,0,1,1,1,1,1,0,1,0,0,0,1,1
sealion,1,0,0,1,0,1,1,1,1,1,0,1,2,1,0,1,1
seasnake,0,0,0,0,0,1,1,1,1,0,1,0,0,1,0,0,3
seawasp,0,0,1,0,0,1,1,0,0,0,1,0,0,0,0,0,7
skimmer,0,1,1,0,1,1,1,0,1,1,0,0,2,1,0,0,2
skua,0,1,1,0,1,1,1,0,1,1,0,0,2,1,0,0,2
slowworm,0,0,1,0,0,0,1,1,1,1,0,0,0,1,0,0,3
slug,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,7
sole,0,0,1,0,0,1,0,1,1,0,0,1,0,1,0,0,4
sparrow,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,0,2
squirrel,1,0,0,1,0,0,0,1,1,1,0,0,2,1,0,0,1
starfish,0,0,1,0,0,1,1,0,0,0,0,0,5,0,0,0,7
stingray,0,0,1,0,0,1,1,1,1,0,1,1,0,1,0,1,4
swan,0,1,1,0,1,1,0,0,1,1,0,0,2,1,0,1,2
termite,0,0,1,0,0,0,0,0,0,1,0,0,6,0,0,0,6
toad,0,0,1,0,0,1,0,1,1,1,0,0,4,0,0,0,5
tortoise,0,0,1,0,0,0,0,0,1,1,0,0,4,1,0,1,3
tuatara,0,0,1,0,0,0,1,1,1,1,0,0,4,1,0,0,3
tuna,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,1,4
vampire,1,0,0,1,1,0,0,1,1,1,0,0,2,1,0,0,1
vole,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,0,1
vulture,0,1,1,0,1,0,1,0,1,1,0,0,2,1,0,1,2
wallaby,1,0,0,1,0,0,0,1,1,1,0,0,2,1,0,1,1
wasp,1,0,1,0,1,0,0,0,0,1,1,0,6,0,0,0,6
wolf,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
worm,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,7
wren,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,0,2"""
    Path("/content/zoo.csv").write_text(zoo.strip() + "\n")

    cls = """Class_Number,Number_Of_Animal_Species_In_Class,Class_Type,Animal_Names
1,41,Mammal,"aardvark, antelope, bear, boar, buffalo, calf, cavy, cheetah, deer, dolphin, elephant, fruitbat, giraffe, girl, goat, gorilla, hamster, hare, leopard, lion, lynx, mink, mole, mongoose, opossum, oryx, platypus, polecat, pony, porpoise, puma, pussycat, raccoon, reindeer, seal, sealion, squirrel, vampire, vole, wallaby, wolf"
2,20,Bird,"chicken, crow, dove, duck, flamingo, gull, hawk, kiwi, lark, ostrich, parakeet, penguin, pheasant, rhea, skimmer, skua, sparrow, swan, vulture, wren"
3,5,Reptile,"pitviper, seasnake, slowworm, tortoise, tuatara"
4,13,Fish,"bass, carp, catfish, chub, dogfish, haddock, herring, pike, piranha, seahorse, sole, stingray, tuna"
5,4,Amphibian,"frog, frog, newt, toad"
6,8,Bug,"flea, gnat, honeybee, housefly, ladybird, moth, termite, wasp"
7,10,Invertebrate,"clam, crab, crayfish, lobster, octopus, scorpion, seawasp, slug, starfish, worm"
"""
    Path("/content/class.csv").write_text(cls.strip() + "\n")

    aux = """[
  {"animal_name":"aardvark","habitat":"savanna","diet":"insectivore","conservation_status":"least concern"},
  {"animal_name":"antelope","habitat":"grasslands","diet":"herbivore","conservation":"near threatened"},
  {"animal_name":"bass","habitat":"freshwater","diet":"carnivore","conservation_status":"least"},
  {"animal_name":"bear","habitats":"forest","diet":"omnivore","conservation_status":"vulnerable"},
  {"animal_name":"boar","habitat":"forest","diet":"omnivor","status":"least concern"},
  {"animal_name":"buffalo","habitat":"grasslands","diet_type":"herbivore","conservation":"endangered"},
  {"animal_name":"calf","habitat":"domestic","diet":"herbivore"},
  {"animal_name":"carp","habitat":"fresh water","diet":"omnivore","conservation_status":"least concern"},
  {"animal_name":"catfish","habitat":"FreshWater","diet":"carnivore"},
  {"animal_name":"clam","habitat":"marine","diet":"filter_feeder","conservation_status":"least concern"},
  {"animal_name":"crab","habitat":"marine/coastal","diet":"omnivore","conservation":"least concern"},
  {"animal_name":"deer","habitat":"Forest","diet":"Herbivore","conservation_status":"vulnerable"}
]"""
    Path("/content/auxiliary_metadata.json").write_text(aux.strip() + "\n")


# --------------------------------------------------------------
# 2. JSON LOADER (robust)
# --------------------------------------------------------------
def _load_aux_json(p: Path) -> List[Dict[str, Any]]:
    out = []
    with p.open(encoding="utf-8") as f:
        try: data = json.load(f)
        except: f.seek(0); data = [json.loads(l) for l in f if l.strip()]

        if not isinstance(data, list): return out
        for rec in data:
            try:
                norm = {k.strip().lower().replace(" ", "_"): v for k, v in rec.items()}
                if "animal_name" not in norm: continue
                for k in {"habitat", "diet", "conservation_status", "conservation"}: norm.setdefault(k, None)
                out.append(norm)
            except: continue
    return out


# --------------------------------------------------------------
# 3. MAIN FUNCTION – TASK 1 + TASK E
# --------------------------------------------------------------
def gamma_load_and_integration(
    zoo_path: str = "/content/zoo.csv",
    class_path: str = "/content/class.csv",
    aux_path: str = "/content/auxiliary_metadata.json"
) -> pd.DataFrame:

    # --- ZOO ---
    zoo_df = pd.read_csv(zoo_path)
    zoo_df.columns = [c.strip().lower() for c in zoo_df.columns]

    # --- CLASS ---
    cls_df = pd.read_csv(class_path)
    cls_df.columns = [c.strip().lower() for c in cls_df.columns]

    cls_expl = (
        cls_df
        .assign(animal_name=cls_df["animal_names"].str.split(", "))
        .explode("animal_name")
        .drop(columns=["animal_names", "number_of_animal_species_in_class"], errors="ignore")
    )
    cls_expl["animal_name"] = cls_expl["animal_name"].astype(str).str.strip().str.lower()
    cls_clean = cls_expl[["animal_name", "class_number", "class_type"]]

    # --- AUX ---
    aux_records = _load_aux_json(Path(aux_path))
    aux_df = pd.DataFrame(aux_records)
    if not aux_df.empty:
        aux_df["animal_name"] = aux_df["animal_name"].astype(str).str.strip().str.lower()

    # --- MERGE ---
    df = zoo_df.copy()
    df["animal_name"] = df["animal_name"].astype(str).str.strip().str.lower()
    df = df.merge(cls_clean, on="animal_name", how="left")
    if not aux_df.empty:
        df = df.merge(aux_df, on="animal_name", how="left")

    # --- FINALIZE ---
    df["animal_name"] = df["animal_name"].str.title()

    # TASK E: Drop rows where ANY auxiliary column is missing
    aux_cols = ["habitat", "diet", "conservation_status", "conservation"]
    aux_cols_present = [col for col in aux_cols if col in df.columns]
    if aux_cols_present:
        print(f"Dropping rows with missing values in: {aux_cols_present}")
        before = len(df)
        df = df.dropna(subset=aux_cols_present, how="any")
        after = len(df)
        print(f"   → {before - after} rows dropped. {after} rows remain.")

    # Reorder columns
    zoo_cols = [c for c in zoo_df.columns if c != "animal_name"]
    final_cols = ["animal_name"] + zoo_cols + ["class_number", "class_type"] + aux_cols_present
    df = df[[c for c in final_cols if c in df.columns]]

    return df


# --------------------------------------------------------------
# 4. RUN EVERYTHING
# --------------------------------------------------------------
print("Creating files in /content ...")
_create_files()

print("\nRunning gamma_load_and_integration() + Task E (drop missing aux data)...\n")
final_df = gamma_load_and_integration()

print("\nFirst 10 rows after dropping missing auxiliary data:\n")
print(final_df.head(10))
print(f"\nFinal shape: {final_df.shape}")
print(f"Columns: {final_df.columns.tolist()}")

Creating files in /content ...

Running gamma_load_and_integration() + Task E (drop missing aux data)...

Dropping rows with missing values in: ['habitat', 'diet', 'conservation_status', 'conservation']
   → 103 rows dropped. 0 rows remain.

First 10 rows after dropping missing auxiliary data:

Empty DataFrame
Columns: [animal_name, hair, feathers, eggs, milk, airborne, aquatic, predator, toothed, backbone, breathes, venomous, fins, legs, tail, domestic, catsize, class_number, habitat, diet, conservation_status, conservation]
Index: []

[0 rows x 22 columns]

Final shape: (0, 22)
Columns: ['animal_name', 'hair', 'feathers', 'eggs', 'milk', 'airborne', 'aquatic', 'predator', 'toothed', 'backbone', 'breathes', 'venomous', 'fins', 'legs', 'tail', 'domestic', 'catsize', 'class_number', 'habitat', 'diet', 'conservation_status', 'conservation']


In [51]:
# --------------------------------------------------------------
#  TASK G – FINAL OUTPUT (Exact Print Statements Required)
# --------------------------------------------------------------

import pandas as pd
import json
from pathlib import Path
from typing import List, Dict, Any

# --------------------------------------------------------------
# 1. CREATE FILES
# --------------------------------------------------------------
def _create_files():
    zoo = """animal_name,hair,feathers,eggs,milk,airborne,aquatic,predator,toothed,backbone,breathes,venomous,fins,legs,tail,domestic,catsize,class_type
aardvark,1,0,0,1,0,0,1,1,1,1,0,0,4,0,0,1,1
antelope,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
bass,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
bear,1,0,0,1,0,0,1,1,1,1,0,0,4,0,0,1,1
boar,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
buffalo,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
calf,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,1,1
carp,0,0,1,0,0,1,0,1,1,0,0,1,0,1,1,0,4
catfish,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
cavy,1,0,0,1,0,0,0,1,1,1,0,0,4,0,1,0,1
cheetah,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
chicken,0,1,1,0,1,0,0,0,1,1,0,0,2,1,1,0,2
chub,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
clam,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,7
crab,0,0,1,0,0,1,1,0,0,0,0,0,4,0,0,0,7
crayfish,0,0,1,0,0,1,1,0,0,0,0,0,6,0,0,0,7
crow,0,1,1,0,1,0,1,0,1,1,0,0,2,1,0,0,2
deer,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
dogfish,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,1,4
dolphin,0,0,0,1,0,1,1,1,1,1,0,1,0,1,0,1,1
dove,0,1,1,0,1,0,0,0,1,1,0,0,2,1,1,0,2
duck,0,1,1,0,1,1,0,0,1,1,0,0,2,1,0,0,2
elephant,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
flamingo,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,1,2
flea,0,0,1,0,0,0,0,0,0,1,0,0,6,0,0,0,6
frog,0,0,1,0,0,1,1,1,1,1,0,0,4,0,0,0,5
frog,0,0,1,0,0,1,1,1,1,1,1,0,4,0,0,0,5
fruitbat,1,0,0,1,1,0,0,1,1,1,0,0,2,1,0,0,1
giraffe,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
girl,1,0,0,1,0,0,1,1,1,1,0,0,2,0,1,1,1
gnat,0,0,1,0,1,0,0,0,0,1,0,0,6,0,0,0,6
goat,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,1,1
gorilla,1,0,0,1,0,0,0,1,1,1,0,0,2,0,0,1,1
gull,0,1,1,0,1,1,1,0,1,1,0,0,2,1,0,0,2
haddock,0,0,1,0,0,1,0,1,1,0,0,1,0,1,0,0,4
hamster,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,0,1
hare,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,0,1
hawk,0,1,1,0,1,0,1,0,1,1,0,0,2,1,0,0,2
herring,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
honeybee,1,0,1,0,1,0,0,0,0,1,1,0,6,0,1,0,6
housefly,1,0,1,0,1,0,0,0,0,1,0,0,6,0,0,0,6
kiwi,0,1,1,0,0,0,1,0,1,1,0,0,2,1,0,0,2
ladybird,0,0,1,0,1,0,1,0,0,1,0,0,6,0,0,0,6
lark,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,0,2
leopard,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
lion,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
lobster,0,0,1,0,0,1,1,0,0,0,0,0,6,0,0,0,7
lynx,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
mink,1,0,0,1,0,1,1,1,1,1,0,0,4,1,0,1,1
mole,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,0,1
mongoose,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
moth,1,0,1,0,1,0,0,0,0,1,0,0,6,0,0,0,6
newt,0,0,1,0,0,1,1,1,1,1,0,0,4,1,0,0,5
octopus,0,0,1,0,0,1,1,0,0,0,0,0,8,0,0,1,7
opossum,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,0,1
oryx,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
ostrich,0,1,1,0,0,0,0,0,1,1,0,0,2,1,0,1,2
parakeet,0,1,1,0,1,0,0,0,1,1,0,0,2,1,1,0,2
penguin,0,1,1,0,0,1,1,0,1,1,0,0,2,1,0,1,2
pheasant,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,0,2
pike,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,1,4
piranha,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
pitviper,0,0,1,0,0,0,1,1,1,1,1,0,0,1,0,0,3
platypus,1,0,1,1,0,1,1,0,1,1,0,0,4,1,0,1,1
polecat,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
pony,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,1,1
porpoise,0,0,0,1,0,1,1,1,1,1,0,1,0,1,0,1,1
puma,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
pussycat,1,0,0,1,0,0,1,1,1,1,0,0,4,1,1,1,1
raccoon,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
reindeer,1,0,0,1,0,0,0,1,1,1,0,0,4,1,1,1,1
rhea,0,1,1,0,0,0,1,0,1,1,0,0,2,1,0,1,2
scorpion,0,0,0,0,0,0,1,0,0,1,1,0,8,1,0,0,7
seahorse,0,0,1,0,0,1,0,1,1,0,0,1,0,1,0,0,4
seal,1,0,0,1,0,1,1,1,1,1,0,1,0,0,0,1,1
sealion,1,0,0,1,0,1,1,1,1,1,0,1,2,1,0,1,1
seasnake,0,0,0,0,0,1,1,1,1,0,1,0,0,1,0,0,3
seawasp,0,0,1,0,0,1,1,0,0,0,1,0,0,0,0,0,7
skimmer,0,1,1,0,1,1,1,0,1,1,0,0,2,1,0,0,2
skua,0,1,1,0,1,1,1,0,1,1,0,0,2,1,0,0,2
slowworm,0,0,1,0,0,0,1,1,1,1,0,0,0,1,0,0,3
slug,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,7
sole,0,0,1,0,0,1,0,1,1,0,0,1,0,1,0,0,4
sparrow,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,0,2
squirrel,1,0,0,1,0,0,0,1,1,1,0,0,2,1,0,0,1
starfish,0,0,1,0,0,1,1,0,0,0,0,0,5,0,0,0,7
stingray,0,0,1,0,0,1,1,1,1,0,1,1,0,1,0,1,4
swan,0,1,1,0,1,1,0,0,1,1,0,0,2,1,0,1,2
termite,0,0,1,0,0,0,0,0,0,1,0,0,6,0,0,0,6
toad,0,0,1,0,0,1,0,1,1,1,0,0,4,0,0,0,5
tortoise,0,0,1,0,0,0,0,0,1,1,0,0,4,1,0,1,3
tuatara,0,0,1,0,0,0,1,1,1,1,0,0,4,1,0,0,3
tuna,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,1,4
vampire,1,0,0,1,1,0,0,1,1,1,0,0,2,1,0,0,1
vole,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,0,1
vulture,0,1,1,0,1,0,1,0,1,1,0,0,2,1,0,1,2
wallaby,1,0,0,1,0,0,0,1,1,1,0,0,2,1,0,1,1
wasp,1,0,1,0,1,0,0,0,0,1,1,0,6,0,0,0,6
wolf,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1
worm,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,7
wren,0,1,1,0,1,0,0,0,1,1,0,0,2,1,0,0,2"""
    Path("/content/zoo.csv").write_text(zoo.strip() + "\n")

    cls = """Class_Number,Number_Of_Animal_Species_In_Class,Class_Type,Animal_Names
1,41,Mammal,"aardvark, antelope, bear, boar, buffalo, calf, cavy, cheetah, deer, dolphin, elephant, fruitbat, giraffe, girl, goat, gorilla, hamster, hare, leopard, lion, lynx, mink, mole, mongoose, opossum, oryx, platypus, polecat, pony, porpoise, puma, pussycat, raccoon, reindeer, seal, sealion, squirrel, vampire, vole, wallaby, wolf"
2,20,Bird,"chicken, crow, dove, duck, flamingo, gull, hawk, kiwi, lark, ostrich, parakeet, penguin, pheasant, rhea, skimmer, skua, sparrow, swan, vulture, wren"
3,5,Reptile,"pitviper, seasnake, slowworm, tortoise, tuatara"
4,13,Fish,"bass, carp, catfish, chub, dogfish, haddock, herring, pike, piranha, seahorse, sole, stingray, tuna"
5,4,Amphibian,"frog, frog, newt, toad"
6,8,Bug,"flea, gnat, honeybee, housefly, ladybird, moth, termite, wasp"
7,10,Invertebrate,"clam, crab, crayfish, lobster, octopus, scorpion, seawasp, slug, starfish, worm"
"""
    Path("/content/class.csv").write_text(cls.strip() + "\n")

    aux = """[
  {"animal_name":"aardvark","habitat":"savanna","diet":"insectivore","conservation_status":"least concern"},
  {"animal_name":"antelope","habitat":"grasslands","diet":"herbivore","conservation":"near threatened"},
  {"animal_name":"bass","habitat":"freshwater","diet":"carnivore","conservation_status":"least"},
  {"animal_name":"bear","habitats":"forest","diet":"omnivore","conservation_status":"vulnerable"},
  {"animal_name":"boar","habitat":"forest","diet":"omnivor","status":"least concern"},
  {"animal_name":"buffalo","habitat":"grasslands","diet_type":"herbivore","conservation":"endangered"},
  {"animal_name":"calf","habitat":"domestic","diet":"herbivore"},
  {"animal_name":"carp","habitat":"fresh water","diet":"omnivore","conservation_status":"least concern"},
  {"animal_name":"catfish","habitat":"FreshWater","diet":"carnivore"},
  {"animal_name":"clam","habitat":"marine","diet":"filter_feeder","conservation_status":"least concern"},
  {"animal_name":"crab","habitat":"marine/coastal","diet":"omnivore","conservation":"least concern"},
  {"animal_name":"deer","habitat":"Forest","diet":"Herbivore","conservation_status":"vulnerable"}
]"""
    Path("/content/auxiliary_metadata.json").write_text(aux.strip() + "\n")


# --------------------------------------------------------------
# 2. LOAD + INTEGRATE + FEATURE ENGINEERING
# --------------------------------------------------------------
def _load_aux_json(p: Path) -> List[Dict[str, Any]]:
    out = []
    with p.open(encoding="utf-8") as f:
        try: data = json.load(f)
        except: f.seek(0); data = [json.loads(l) for l in f if l.strip()]
        if not isinstance(data, list): return out
        for rec in data:
            try:
                norm = {k.strip().lower().replace(" ", "_"): v for k, v in rec.items()}
                if "animal_name" not in norm: continue
                for k in {"habitat","diet","conservation_status","conservation"}: norm.setdefault(k, None)
                out.append(norm)
            except: continue
    return out

def load_and_merge():
    # Load zoo
    zoo_df = pd.read_csv("/content/zoo.csv")
    zoo_df.columns = [c.strip().lower() for c in zoo_df.columns]

    # Load class & explode
    cls_df = pd.read_csv("/content/class.csv")
    cls_df.columns = [c.strip().lower() for c in cls_df.columns]
    cls_expl = (cls_df
                .assign(animal_name=cls_df["animal_names"].str.split(", "))
                .explode("animal_name")
                .drop(columns=["animal_names","number_of_animal_species_in_class"], errors="ignore"))
    cls_expl["animal_name"] = cls_expl["animal_name"].astype(str).str.strip().str.lower()
    cls_clean = cls_expl[["animal_name","class_number","class_type"]]

    # Load aux
    aux_records = _load_aux_json(Path("/content/auxiliary_metadata.json"))
    aux_df = pd.DataFrame(aux_records)
    if not aux_df.empty:
        aux_df["animal_name"] = aux_df["animal_name"].astype(str).str.strip().str.lower()

    # Merge
    df = zoo_df.copy()
    df["animal_name"] = df["animal_name"].str.strip().str.lower()
    df = df.merge(cls_clean, on="animal_name", how="left")
    if not aux_df.empty:
        df = df.merge(aux_df, on="animal_name", how="left")

    # Task E: Drop rows with missing aux data
    aux_cols = ["habitat","diet","conservation_status","conservation"]
    aux_present = [c for c in aux_cols if c in df.columns]
    df = df.dropna(subset=aux_present, how="any")

    # Task F: Feature Engineering
    priority_map = {"endangered":5,"vulnerable":4,"near threatened":3,"least":1,"least concern":1}
    def get_priority(v):
        if pd.isna(v): return 0
        v = str(v).lower()
        for k, s in priority_map.items():
            if k in v: return s
        return 0
    df["conservation_priority"] = df["conservation_status"].fillna(df["conservation"]).apply(get_priority)
    df["aquatic_flag"] = df["habitat"].astype(str).str.lower().apply(lambda x: 1 if "water" in x or "marine" in x else 0)

    df["animal_name"] = df["animal_name"].str.title()
    return df


# --------------------------------------------------------------
# 3. RUN & PRINT FINAL OUTPUT (Task G)
# --------------------------------------------------------------
print("Creating files...")
_create_files()

print("\nLoading, merging, and engineering features...")
merged_data = load_and_merge()

# Engineered feature names
engineered_feature_name = ["conservation_priority", "aquatic_flag"]

# === REQUIRED OUTPUT ===
print(f"dataset shape:{merged_data.shape}")
print(f"missing value : {merged_data.isnull().sum().sum()}")
print(f"duplicate rows: {merged_data.duplicated().sum()}")
print("\nfirst 3 rows:")
print(merged_data.head(3))
print(f"\nengineered features: {engineered_feature_name}")

Creating files...

Loading, merging, and engineering features...
dataset shape:(0, 29)
missing value : 0
duplicate rows: 0

first 3 rows:
Empty DataFrame
Columns: [animal_name, hair, feathers, eggs, milk, airborne, aquatic, predator, toothed, backbone, breathes, venomous, fins, legs, tail, domestic, catsize, class_type_x, class_number, class_type_y, habitat, diet, conservation_status, conservation, habitats, status, diet_type, conservation_priority, aquatic_flag]
Index: []

[0 rows x 29 columns]

engineered features: ['conservation_priority', 'aquatic_flag']
